# Folder 03 / file 03 — canary_start / metrics_gate / to_100

UCI Adult Census Income ([dataset](https://archive.ics.uci.edu/dataset/2/adult)): binary target `income_gt_50k` where **`>50K` = 1** and **`<=50K` = 0**. Fourteen census features; official split is `adult.data` (train) / `adult.test` (holdout).

Inlined replica of `src/n03_prod_cutover/n03_canary.py`. Run cells **in order** (local or Jobs). Catalog/schema/model/mode come from task env.

Optional canary traffic for the Adult prod endpoint (does not flip @prod until to_100).


## 1 — Imports


In [ ]:
STOP = False

def _stop(msg: str = "") -> None:
    global STOP
    STOP = True
    print(msg)
    try:
        dbutils.notebook.exit(msg or "ok")  # noqa: F821
    except Exception:
        pass

import time
from src.n03_prod_cutover.n01_approval import resolve_cutover_dest_version
from src.n00_shared.runtime import Settings, configure_mlflow, current_task_key, load_settings, mlflow_client
from src.n00_shared.serving import serving_config, upsert_endpoint, wait_endpoint_ready


## 2 — `_tag_map`


In [ ]:
def _tag_map(client, name: str, version: str) -> dict:
    mv = client.get_model_version(name, version)
    tags = mv.tags or {}
    if isinstance(tags, dict):
        return tags
    return {t.key: t.value for t in tags}


## 3 — `settings = load_settings()`


In [ ]:
settings = load_settings()


## 4 — `run()` step 1/2


In [ ]:
if not STOP:
    configure_mlflow(settings)
    task = current_task_key() or __import__("os").environ.get("CANARY_ACTION", "")
    client = mlflow_client()
    dest = settings.dest_model_name
    version = resolve_cutover_dest_version(settings)
    tags = _tag_map(client, dest, version)
    previous = str(tags.get("previous_prod_version") or "")
    if not settings.allow_canary or not previous:
        print(f"{task or 'canary'}: skipped (allow_canary={settings.allow_canary} previous={previous})")
        _stop()
    if task in {"", "canary_start", "canary"}:
        upsert_endpoint(
            settings.endpoint_name,
            serving_config(
                dest,
                version,
                previous_version=previous,
                canary_percent=settings.canary_percent,
            ),
        )
        time.sleep(min(settings.canary_warmup_s, 30))
        print(f"canary_start {settings.canary_percent}% on v{version}")
        _stop()
    if task == "metrics_gate":
        wait_endpoint_ready(settings.endpoint_name)
        print("metrics_gate: endpoint READY (wire live canary SLOs when traffic exists)")
        _stop()


## 5 — `run()` step 2/2


In [ ]:
if not STOP:
    if task == "to_100":
        client.set_registered_model_alias(dest, "prod", version)
        client.set_registered_model_alias(dest, "champion", version)
        upsert_endpoint(
            settings.endpoint_name,
            serving_config(dest, version, previous_version=None, canary_percent=100),
        )
        print(f"to_100 set @prod/@champion and serving 100% to v{version}")
        _stop()
    raise RuntimeError(f"unknown canary task {task}")
